#### **3. Optimization of Logistic Regression**

In [1]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Two Gaussian blobs, one per class
X, y = make_blobs(
    n_samples=1200,
    centers=[[-2.2, -0.5], [2.0, 1.2]],
    cluster_std=[1.45, 1.65],
    random_state=8
)

# Stratified 80:20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}, test: {X_test.shape}")
print(f"Positive class share - train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

Train: (960, 2), test: (240, 2)
Positive class share - train: 0.500, test: 0.500


In [2]:
import matplotlib.pyplot as plt
import os

os.makedirs('figures', exist_ok=True)

# Figure output quality settings
plt.rcParams['figure.dpi']     = 120    # on-screen preview
plt.rcParams['savefig.dpi']    = 500    # saved PNG files
plt.rcParams['savefig.bbox']   = 'tight'
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.25
plt.rcParams['grid.linewidth'] = 0.6


def save_fig(name):
    """Save the current figure to figures/<name>.png at 500 dpi."""
    plt.savefig(f'figures/{name}.png', bbox_inches='tight')

*Notation used throughout this section*

Let $\mathbf{X} \in \mathbb{R}^{N \times (D+1)}$ be the augmented feature matrix including a bias column of ones, $\mathbf{y} \in \{0, 1\}^N$ be target labels, $\mathbf{w} \in \mathbb{R}^{D+1}$ be model parameters, and

$$p_i = \sigma(\mathbf{w}^T\mathbf{x}_i) = \frac{1}{1 + e^{-\mathbf{w}^T\mathbf{x}_i}}.$$

High-level estimators such as `sklearn.linear_model.LogisticRegression` are not used for questions 1-6 in this section.


---
**Question 1**

Derive the gradient vector $\nabla J(\mathbf{w})$ and Hessian matrix $\mathbf{H}$ for Binary Cross-Entropy loss:

$$J(\mathbf{w}) = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \ln p_i + (1 - y_i)\ln(1 - p_i)\right].$$

---


$$J(\mathbf{w}) = -\frac{1}{N}\sum_{i=1}^{N}\big[y_i\ln p_i + (1-y_i)\ln(1-p_i)\big],
\qquad p_i = \sigma(z_i),\quad z_i = \mathbf{w}^{T}\mathbf{x}_i$$

with $\mathbf{x}_i \in \mathbb{R}^{D+1}$ including the lead $1$ for the bias, and
$\sigma'(z) = \sigma(z)(1-\sigma(z))$.

### Gradient

Per-sample loss $\ell_i = -y_i\ln p_i - (1-y_i)\ln(1-p_i)$. Differentiating w.r.t. $p_i$:

$$\frac{\partial \ell_i}{\partial p_i} = -\frac{y_i}{p_i} + \frac{1-y_i}{1-p_i}$$

Chaining through $p_i = \sigma(z_i)$ and $z_i = \mathbf{w}^{T}\mathbf{x}_i$, where
$\dfrac{\partial p_i}{\partial\mathbf{w}} = p_i(1-p_i)\mathbf{x}_i$:

$$\nabla J(\mathbf{w}) = \frac{1}{N}\sum_{i=1}^{N}
\left[-\frac{y_i}{p_i} + \frac{1-y_i}{1-p_i}\right]p_i(1-p_i)\,\mathbf{x}_i$$

The bracket collapses:

$$\left[-\frac{y_i}{p_i} + \frac{1-y_i}{1-p_i}\right]p_i(1-p_i)
= -y_i(1-p_i) + (1-y_i)p_i = p_i - y_i$$

$$\boxed{\;\nabla J(\mathbf{w}) = \frac{1}{N}\sum_{i=1}^{N}(p_i - y_i)\mathbf{x}_i
= \frac{1}{N}\mathbf{X}^{T}(\mathbf{p} - \mathbf{y})\;}$$

where $\mathbf{p} = \sigma(\mathbf{Xw})$ elementwise.

### Hessian

Only $p_i$ depends on $\mathbf{w}$, and
$\dfrac{\partial p_i}{\partial\mathbf{w}^{T}} = p_i(1-p_i)\mathbf{x}_i^{T}$, so

$$\mathbf{H} = \frac{\partial}{\partial\mathbf{w}^{T}}
\left[\frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)\mathbf{x}_i\right]
= \frac{1}{N}\sum_{i=1}^{N}\mathbf{x}_i\,\frac{\partial p_i}{\partial\mathbf{w}^{T}}$$

$$\boxed{\;\mathbf{H} = \frac{1}{N}\sum_{i=1}^{N}p_i(1-p_i)\,\mathbf{x}_i\mathbf{x}_i^{T}
= \frac{1}{N}\mathbf{X}^{T}\mathbf{S}\mathbf{X}\;}$$

with $\mathbf{S} = \operatorname{diag}(s_1,\dots,s_N)$, $s_i = p_i(1-p_i)$. Note
$\mathbf{H} \in \mathbb{R}^{(D+1)\times(D+1)}$.

### Convexity

For any $\mathbf{v} \in \mathbb{R}^{D+1}$,

$$\mathbf{v}^{T}\mathbf{H}\mathbf{v}
= \frac{1}{N}(\mathbf{Xv})^{T}\mathbf{S}(\mathbf{Xv})
= \frac{1}{N}\sum_{i=1}^{N}s_i\big(\mathbf{x}_i^{T}\mathbf{v}\big)^{2} \;\ge\; 0$$

since $s_i = p_i(1-p_i) > 0$ for all finite $z_i$. Hence $\mathbf{H} \succeq 0$ and $J$ is
convex, so **any stationary point is a global minimum** ,the minimiser is unique
(strict convexity) when $\mathbf{X}$ has full column rank $D+1$.

---
**Question 2**

Write Python functions for numerically stable sigmoid activation and cross-entropy loss computation. Explain why raw implementations of $\ln(p)$ fail when $p \to 0$ or $p \to 1$, and why techniques like clipping probabilities ($\varepsilon = 10^{-15}$) or utilizing `np.logaddexp` prevent underflow/overflow.

---


In [3]:
def sigmoid(z):
    """Numerically stable logistic sigmoid."""
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0

    # z >= 0: use 1/(1+e^-z), where e^-z <= 1 so it cannot overflow
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))

    # z < 0: use e^z/(1+e^z), where e^z <= 1 so it cannot overflow either
    e = np.exp(z[~pos])
    out[~pos] = e / (1.0 + e)
    return out


def bce_loss(z, y):
    """Binary cross-entropy from logits z, without ever forming p."""
    z = np.asarray(z, dtype=float)
    # ln(1+e^z) - y*z  is the whole per-sample loss (see the answer below)
    return float(np.mean(np.logaddexp(0.0, z) - y * z))


def logits(X, w):
    """Linear scores z = Xw for the augmented design matrix X."""
    return X @ w


# Quick check against the closed-form values at z = 0
print(f"sigmoid(0)          = {sigmoid(np.array([0.0]))[0]:.4f}   (expect 0.5)")
print(f"bce_loss(0, y=1)    = {bce_loss(np.array([0.0]), np.array([1.0])):.4f}   "
      f"(expect ln 2 = {np.log(2):.4f})")

sigmoid(0)          = 0.5000   (expect 0.5)
bce_loss(0, y=1)    = 0.6931   (expect ln 2 = 0.6931)


In binary cross-entropy, we calculate

$$-\left[y\ln(p)+(1-y)\ln(1-p)\right]$$

The problem happens when $p$ gets very close to 0 or 1. Because of floating-point precision, the sigmoid can become exactly 0 or 1 for very large positive or negative values of $z$.

For example, if $p=0$,

$$\ln(0)=-\infty$$

This can make the loss become `inf` and may also cause `NaN` values during gradient calculations.

There is also a problem with calculating the sigmoid directly as

$$\sigma(z)=\frac{1}{1+e^{-z}}$$

For a large negative $z$, such as $z=-800$,

$$e^{-z}=e^{800}$$

is too large for a float64 number, so it overflows.

**Clipping the probabilities**

One simple solution is to clip the probability:

```python
p = np.clip(p, 1e-15, 1 - 1e-15)
```

This means $p$ is never exactly 0 or 1, so the logarithm stays finite.

However, clipping slightly changes the true value. For example,

$$-\ln(10^{-15}) \approx 34.54$$

So if the real loss is much larger than this, clipping will give a smaller value instead.

Thus, clipping is easy to use, but it can slightly affect the result for extreme